# Hemorrhage and resuscitation in the Pulse engine

_Investigation `pulse-hemorrhage` — coder reproduction notebook._

**Question.** How does the whole-body Pulse engine respond to a hemorrhage: how does the response scale with bleed severity, does closed-loop fluid resuscitation change it, and does the same bleed behave the same across Pulse's 21 shipped patients?

Three workbench studies of hemorrhage in the real Pulse whole-body engine.

---

**How to use this notebook**

1. **Run ▸ Run All Cells** to reproduce every study end to end.
2. To explore, edit a study's **Parameters** cell, then re-run from its **Run** section — the figures regenerate with your values.
3. The **Setup** cell at the top is collapsed scaffolding (environment + render helpers); expand it only to inspect. Set `RERUN = False` there to render the committed `runs.db` without re-simulating.

The prose states the **question and parameters**; the figures each run produces are the results.


In [ ]:
"""Setup — environment & helpers (runs automatically; open only to inspect).

Generated by vivarium-workbench (notebook_export). This cell resolves the repo,
registers the workspace Core, and defines the render helpers; the science lives
in the sections below. Each study is re-run live with the workspace's own
process-bigraph protocol and its figures rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-pulse/viva-pulse').is_dir():
    REPO = Path('/home/runner/work/viva-pulse/viva-pulse')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# Keep the output clean: composite discovery and the pydantic models emit
# benign UserWarnings (optional packages it skips, a field-name shadow) and the
# simulators emit DeprecationWarnings — none are actionable from a reproduction
# notebook, so silence those two categories. Remove this to see them.
import warnings as _warnings
_warnings.filterwarnings("ignore", category=UserWarning)
_warnings.filterwarnings("ignore", category=DeprecationWarning)

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_pulse.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

def materialize_composite(spec_id, overrides=None):
    """Build a composite into an editable payload {name, description,
    parameters, state, _overrides, _id}. Works for @composite_generator
    functions via the process-bigraph registry (the common case); ``state``
    is the built document (process nodes at top level)."""
    import importlib
    from process_bigraph.composite_generator import (
        _REGISTRY, apply_core_extensions, build_generator)
    try:
        from process_bigraph.composite_generator import discover_generators
        discover_generators()
    except Exception:
        pass
    entry = _REGISTRY.get(spec_id)
    if entry is None:
        # editable installs are invisible to packages_distributions, so
        # discover alone can miss them — prime the module the id names.
        try:
            importlib.import_module(spec_id.rsplit('.', 1)[0])
            from process_bigraph.composite_generator import discover_generators as _dg
            _dg()
        except Exception:
            pass
        entry = _REGISTRY.get(spec_id)
    if entry is None:
        raise KeyError('composite not registered: ' + repr(spec_id))
    params = dict(entry.parameters or {})
    ov = {k: (v.get('default') if isinstance(v, dict) else None)
          for k, v in params.items()}
    if overrides:
        ov.update({k: val for k, val in overrides.items() if k in params})
    _c = apply_core_extensions(entry, core)
    try:
        doc = build_generator(entry, ov or None, core=_c)
    except TypeError:
        doc = entry.func(_c, **(ov or {}))
    return {'name': entry.name, 'description': entry.description,
            'parameters': params, 'state': doc, '_overrides': ov, '_id': spec_id}

import base64 as _b64, pathlib as _pl
_STATIC_VIZ_SCHEMES = ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:')
def _render_viz_process(address, config):
    """Render a Visualization process/step (e.g. ``local:<Class>``) to
    HTML: wrap it as a one-step composite and run it once — the viz writes
    Plotly HTML to its ``html`` output port. Mirrors the workbench's own
    env-worker viz path. Self-running vizes need no run data."""
    from process_bigraph import Composite
    _doc = {
        'inputs_store': {}, 'output_store': '',
        'visualization': {
            '_type': 'step', 'address': address,
            'config': {k: v for k, v in (config or {}).items()
                       if k not in ('inputs_map', 'sources')},
            'inputs': {}, 'outputs': {'html': ['output_store']},
        },
    }
    _comp = Composite({'state': _doc}, core=core)
    _comp.run(1)
    _html = _comp.state.get('output_store')
    if isinstance(_html, dict):
        _html = _html.get('value') or _html.get('_value') or ''
    return _html if isinstance(_html, str) else ''
def _render_one(address, config, runs_db, study_yaml, overrides=None):
    """Generic figure renderer (no workspace render_study_viz.py).
    A static ``image:``/``file:`` (or bare-path) address embeds the file;
    any other address (``local:<VizProcess>``) is run as a Visualization
    and its HTML returned. ``overrides`` (the composite's edited parameter
    dict) is merged into the viz config first so a self-running figure
    reflects edited parameters; a viz that does not accept those keys
    falls back to the base config."""
    addr = str(address or '')
    _head = addr.split('/', 1)[0]
    _is_static = (not addr) or (':' not in _head) or addr.startswith(_STATIC_VIZ_SCHEMES)
    if not _is_static:
        if core is None:
            return f'<p style="color:#b91c1c">no core to render {address}</p>'
        _base = {k: v for k, v in (config or {}).items()}
        _tries = ([{**_base, **overrides}] if overrides else []) + [_base]
        _err = ''
        for _cfg in _tries:
            try:
                _h = _render_viz_process(addr, _cfg)
            except Exception as _e:
                _err = (f'<p style="color:#b91c1c">visualization {address} not renderable yet: '
                        f'{type(_e).__name__}: {_e}</p>')
                continue
            return _h or f'<p style="color:#6b7280">{address} produced no HTML</p>'
        return _err
    for _scheme in _STATIC_VIZ_SCHEMES:
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Hemorrhage severity sweep (`hemorrhage-severity-sweep`)

**Question.** How do blood loss, heart rate and mean arterial pressure over 10 minutes
depend on the severity of a right-leg hemorrhage in the StandardMale patient?

**Objective.** Run pulse_hemorrhage for 600 s at severity 0.0, 0.2, 0.4, 0.6, 0.8 and 1.0
(RightLeg, StandardMale) and compare the final vitals across the grid.

**Hypothesis.** Blood lost and heart rate rise monotonically with severity, while mean
arterial pressure stays within 2 mmHg of its starting value for mild bleeds
(the baroreflex compensates by raising heart rate) and falls only once the
bleed is severe enough to outrun that compensation.

**Claim.** Blood loss and heart rate rise steadily with severity, while MAP ends within 1.1 mmHg of its start, and never strays more than 1.8 mmHg, up to severity 0.4 and falls 7-17 mmHg from severity 0.6 upward.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `severity-0.0` | `viva_pulse.composites.hemorrhage.pulse_hemorrhage` | 600 | severity=0.0 |
| `severity-0.2` | `viva_pulse.composites.hemorrhage.pulse_hemorrhage` | 600 | severity=0.2 |
| `severity-0.4` | `viva_pulse.composites.hemorrhage.pulse_hemorrhage` | 600 | severity=0.4 |
| `severity-0.6` | `viva_pulse.composites.hemorrhage.pulse_hemorrhage` | 600 | severity=0.6 |
| `severity-0.8` | `viva_pulse.composites.hemorrhage.pulse_hemorrhage` | 600 | state_file=StandardMale@0s.json, compartment=RightLeg, severity=0.8 |
| `severity-1.0` | `viva_pulse.composites.hemorrhage.pulse_hemorrhage` | 600 | severity=1.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_pulse.composites.hemorrhage.pulse_hemorrhage`**

Untreated hemorrhage in the real Pulse whole-body adult engine; emits vital signs every second.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage = {
    'state_file'       : 'StandardMale@0s.json',   # Stabilized Pulse patient state in <install>/bin/states
    'patient_file'     : 'StandardMale.json',   # Any patient instead of state_file: a name in <install>/bin/patients or a path; stabilized once (about 13 s), then cached
    'circuit_modifiers': {},   # Circuit path or node name -> multiplier on the value Pulse initializes it with; the patient is stabilized with them (about 25 s the first time, then cached)
    'circuit_additions': {},   # New path name -> {source, target, resistance_mmHg_s_per_mL}: a cardiovascular path added with its compartment link, as a perturbation of the loaded patient
    'compartment'      : 'RightLeg',   # Hemorrhage site (Pulse eHemorrhage_Compartment)
    'severity'         : 0.8,   # Hemorrhage severity in [0, 1]
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: hemorrhage-severity-sweep ===
STUDY = 'hemorrhage-severity-sweep'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Reproduce the recorded run(s). STEPS = composite steps.
STEPS_severity_0_0 = 600
INTERVAL_severity_0_0 = 1.0
STEPS_severity_0_2 = 600
INTERVAL_severity_0_2 = 1.0
STEPS_severity_0_4 = 600
INTERVAL_severity_0_4 = 1.0
STEPS_severity_0_6 = 600
INTERVAL_severity_0_6 = 1.0
STEPS_severity_0_8 = 600
INTERVAL_severity_0_8 = 1.0
STEPS_severity_1_0 = 600
INTERVAL_severity_1_0 = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage if 'ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage = materialize_composite('viva_pulse.composites.hemorrhage.pulse_hemorrhage', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage['state']}, core=core)
            _comp.run(STEPS_severity_0_0)
        print('ran', 'severity-0.0', f'(viva_pulse.composites.hemorrhage.pulse_hemorrhage) for', STEPS_severity_0_0, 'steps')
    except Exception as _e:
        print(f"severity-0.0 not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage if 'ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage = materialize_composite('viva_pulse.composites.hemorrhage.pulse_hemorrhage', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage['state']}, core=core)
            _comp.run(STEPS_severity_0_2)
        print('ran', 'severity-0.2', f'(viva_pulse.composites.hemorrhage.pulse_hemorrhage) for', STEPS_severity_0_2, 'steps')
    except Exception as _e:
        print(f"severity-0.2 not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage if 'ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage = materialize_composite('viva_pulse.composites.hemorrhage.pulse_hemorrhage', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage['state']}, core=core)
            _comp.run(STEPS_severity_0_4)
        print('ran', 'severity-0.4', f'(viva_pulse.composites.hemorrhage.pulse_hemorrhage) for', STEPS_severity_0_4, 'steps')
    except Exception as _e:
        print(f"severity-0.4 not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage if 'ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage = materialize_composite('viva_pulse.composites.hemorrhage.pulse_hemorrhage', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage['state']}, core=core)
            _comp.run(STEPS_severity_0_6)
        print('ran', 'severity-0.6', f'(viva_pulse.composites.hemorrhage.pulse_hemorrhage) for', STEPS_severity_0_6, 'steps')
    except Exception as _e:
        print(f"severity-0.6 not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage if 'ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage = materialize_composite('viva_pulse.composites.hemorrhage.pulse_hemorrhage', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage['state']}, core=core)
            _comp.run(STEPS_severity_0_8)
        print('ran', 'severity-0.8', f'(viva_pulse.composites.hemorrhage.pulse_hemorrhage) for', STEPS_severity_0_8, 'steps')
    except Exception as _e:
        print(f"severity-0.8 not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage if 'ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage = materialize_composite('viva_pulse.composites.hemorrhage.pulse_hemorrhage', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage['state']}, core=core)
            _comp.run(STEPS_severity_1_0)
        print('ran', 'severity-1.0', f'(viva_pulse.composites.hemorrhage.pulse_hemorrhage) for', STEPS_severity_1_0, 'steps')
    except Exception as _e:
        print(f"severity-1.0 not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**severity-response**


In [ ]:
# severity-response
show_viz(_render_one('local:HemorrhageSeverityResponse', {'inputs_map': {'time': 'global_time', 'severity': 'interventions.hemorrhage_severity', 'mean_arterial_pressure': 'vitals.mean_arterial_pressure_mmHg', 'heart_rate': 'vitals.heart_rate_bpm', 'hemorrhaged_volume': 'vitals.total_hemorrhaged_volume_mL'}}, RUNS_DB, STUDY_YAML))

## Study: Fluid resuscitation after hemorrhage (`fluid-resuscitation`)

**Question.** After a severe right-leg hemorrhage, does a closed-loop MAP-targeted infusion
preserve blood volume and arterial pressure compared with no treatment, and
does the infused compound (saline vs whole blood) matter?

**Objective.** Run the same bleed (StandardMale, RightLeg, severity 0.8, 600 s) untreated
(pulse_hemorrhage), with the MAP-targeted saline controller
(pulse_resuscitation, target 90 mmHg, gain 25 mL/min per mmHg, cap 250 mL/min)
and with the same controller infusing whole blood.

**Hypothesis.** Both treated arms end the 10 minutes with more blood volume and a higher mean
arterial pressure than the untreated arm, because the controller infuses
whenever MAP falls below its 90 mmHg target.

**Claim.** Both treated arms end with more blood volume (+78 mL) and a higher MAP (+2.2 mmHg) than the untreated arm, but saline and whole blood are indistinguishable over 10 minutes.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `blood-map-90` | `viva_pulse.composites.hemorrhage.pulse_resuscitation` | 600 | state_file=StandardMale@0s.json, compartment=RightLeg, severity=0.8, compound=Blood, target_map_mmHg=90.0, gain=25.0, max_rate_mL_per_min=250.0 |
| `saline-map-90` | `viva_pulse.composites.hemorrhage.pulse_resuscitation` | 600 | state_file=StandardMale@0s.json, compartment=RightLeg, severity=0.8, compound=Saline, target_map_mmHg=90.0, gain=25.0, max_rate_mL_per_min=250.0 |
| `untreated` | `viva_pulse.composites.hemorrhage.pulse_hemorrhage` | 600 | state_file=StandardMale@0s.json, compartment=RightLeg, severity=0.8 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_pulse.composites.hemorrhage.pulse_resuscitation`**

Hemorrhage in the real Pulse engine with a closed-loop MAP-targeted IV fluid controller.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_pulse_composites_hemorrhage_pulse_resuscitation = {
    'state_file'         : 'StandardMale@0s.json',   # Stabilized Pulse patient state in <install>/bin/states
    'patient_file'       : 'StandardMale.json',   # Any patient instead of state_file: a name in <install>/bin/patients or a path; stabilized once (about 13 s), then cached
    'circuit_modifiers'  : {},   # Circuit path or node name -> multiplier on the value Pulse initializes it with; the patient is stabilized with them (about 25 s the first time, then cached)
    'circuit_additions'  : {},   # New path name -> {source, target, resistance_mmHg_s_per_mL}: a cardiovascular path added with its compartment link, as a perturbation of the loaded patient
    'compartment'        : 'RightLeg',   # Hemorrhage site (Pulse eHemorrhage_Compartment)
    'severity'           : 0.8,   # Hemorrhage severity in [0, 1]
    'compound'           : 'Saline',   # Infused compound (Pulse compound name, e.g. Saline, Blood)
    'target_map_mmHg'    : 90.0,   # Controller MAP target
    'gain'               : 25.0,   # Controller gain, mL/min per mmHg below target
    'max_rate_mL_per_min': 250.0,   # Infusion rate ceiling
}

**The model — `viva_pulse.composites.hemorrhage.pulse_hemorrhage`**

Untreated hemorrhage in the real Pulse whole-body adult engine; emits vital signs every second.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage = {
    'state_file'       : 'StandardMale@0s.json',   # Stabilized Pulse patient state in <install>/bin/states
    'patient_file'     : 'StandardMale.json',   # Any patient instead of state_file: a name in <install>/bin/patients or a path; stabilized once (about 13 s), then cached
    'circuit_modifiers': {},   # Circuit path or node name -> multiplier on the value Pulse initializes it with; the patient is stabilized with them (about 25 s the first time, then cached)
    'circuit_additions': {},   # New path name -> {source, target, resistance_mmHg_s_per_mL}: a cardiovascular path added with its compartment link, as a perturbation of the loaded patient
    'compartment'      : 'RightLeg',   # Hemorrhage site (Pulse eHemorrhage_Compartment)
    'severity'         : 0.8,   # Hemorrhage severity in [0, 1]
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fluid-resuscitation ===
STUDY = 'fluid-resuscitation'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Reproduce the recorded run(s). STEPS = composite steps.
STEPS_blood_map_90 = 600
INTERVAL_blood_map_90 = 1.0
STEPS_saline_map_90 = 600
INTERVAL_saline_map_90 = 1.0
STEPS_untreated = 600
INTERVAL_untreated = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_resuscitation if 'ov_viva_pulse_composites_hemorrhage_pulse_resuscitation' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_resuscitation = materialize_composite('viva_pulse.composites.hemorrhage.pulse_resuscitation', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_resuscitation['state']}, core=core)
            _comp.run(STEPS_blood_map_90)
        print('ran', 'blood-map-90', f'(viva_pulse.composites.hemorrhage.pulse_resuscitation) for', STEPS_blood_map_90, 'steps')
    except Exception as _e:
        print(f"blood-map-90 not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_resuscitation if 'ov_viva_pulse_composites_hemorrhage_pulse_resuscitation' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_resuscitation = materialize_composite('viva_pulse.composites.hemorrhage.pulse_resuscitation', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_resuscitation['state']}, core=core)
            _comp.run(STEPS_saline_map_90)
        print('ran', 'saline-map-90', f'(viva_pulse.composites.hemorrhage.pulse_resuscitation) for', STEPS_saline_map_90, 'steps')
    except Exception as _e:
        print(f"saline-map-90 not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage if 'ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage = materialize_composite('viva_pulse.composites.hemorrhage.pulse_hemorrhage', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage['state']}, core=core)
            _comp.run(STEPS_untreated)
        print('ran', 'untreated', f'(viva_pulse.composites.hemorrhage.pulse_hemorrhage) for', STEPS_untreated, 'steps')
    except Exception as _e:
        print(f"untreated not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**resuscitation-response**


In [ ]:
# resuscitation-response
show_viz(_render_one('local:FluidResuscitationResponse', {'inputs_map': {'time': 'global_time', 'mean_arterial_pressure': 'vitals.mean_arterial_pressure_mmHg', 'blood_volume': 'vitals.blood_volume_mL', 'iv_rate': 'interventions.iv_rate_mL_per_min'}}, RUNS_DB, STUDY_YAML))

## Study: Hemorrhage across the shipped patients (`patient-cohort`)

**Question.** Does the same right-leg hemorrhage (severity 0.8, 10 minutes) produce the
same qualitative response in all 21 patients that ship with Pulse, and how
does the fraction of blood lost depend on the patient's starting blood volume?

**Objective.** Run pulse_hemorrhage (RightLeg, severity 0.8, 600 s) once from each of the 21
stabilized states in Pulse's install (bin/states/*@0s.json) and compare the
final vitals and the fraction of blood lost.

**Hypothesis.** Every patient loses blood volume, raises heart rate and loses arterial pressure. Absolute blood loss varies much less than starting blood volume does, so patients with a smaller starting blood volume lose a larger fraction of it.

**Claim.** All 21 shipped patients bleed and compensate the same way, and the fraction of blood lost falls with starting blood volume (Spearman rho = -0.89).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Bradycardic` | `viva_pulse.composites.hemorrhage.pulse_hemorrhage` | 600 | state_file=Bradycardic@0s.json |
| `Carol` | `viva_pulse.composites.hemorrhage.pulse_hemorrhage` | 600 | state_file=Carol@0s.json |
| `Cynthia` | `viva_pulse.composites.hemorrhage.pulse_hemorrhage` | 600 | state_file=Cynthia@0s.json |
| `DefaultFemale` | `viva_pulse.composites.hemorrhage.pulse_hemorrhage` | 600 | state_file=DefaultFemale@0s.json |
| `DefaultMale` | `viva_pulse.composites.hemorrhage.pulse_hemorrhage` | 600 | state_file=DefaultMale@0s.json |
| `ExtremeFemale` | `viva_pulse.composites.hemorrhage.pulse_hemorrhage` | 600 | state_file=ExtremeFemale@0s.json |
| `ExtremeMale` | `viva_pulse.composites.hemorrhage.pulse_hemorrhage` | 600 | state_file=ExtremeMale@0s.json |
| `Gus` | `viva_pulse.composites.hemorrhage.pulse_hemorrhage` | 600 | state_file=Gus@0s.json |
| `Hassan` | `viva_pulse.composites.hemorrhage.pulse_hemorrhage` | 600 | state_file=Hassan@0s.json |
| `Jane` | `viva_pulse.composites.hemorrhage.pulse_hemorrhage` | 600 | state_file=Jane@0s.json |
| `Jeff` | `viva_pulse.composites.hemorrhage.pulse_hemorrhage` | 600 | state_file=Jeff@0s.json |
| `Joel` | `viva_pulse.composites.hemorrhage.pulse_hemorrhage` | 600 | state_file=Joel@0s.json |
| `Nathan` | `viva_pulse.composites.hemorrhage.pulse_hemorrhage` | 600 | state_file=Nathan@0s.json |
| `Overweight` | `viva_pulse.composites.hemorrhage.pulse_hemorrhage` | 600 | state_file=Overweight@0s.json |
| `Rick` | `viva_pulse.composites.hemorrhage.pulse_hemorrhage` | 600 | state_file=Rick@0s.json |
| `Soldier` | `viva_pulse.composites.hemorrhage.pulse_hemorrhage` | 600 | state_file=Soldier@0s.json |
| `StandardFemale` | `viva_pulse.composites.hemorrhage.pulse_hemorrhage` | 600 | state_file=StandardFemale@0s.json |
| `StandardMale` | `viva_pulse.composites.hemorrhage.pulse_hemorrhage` | 600 | state_file=StandardMale@0s.json, compartment=RightLeg, severity=0.8 |
| `Tachycardic` | `viva_pulse.composites.hemorrhage.pulse_hemorrhage` | 600 | state_file=Tachycardic@0s.json |
| `Underweight` | `viva_pulse.composites.hemorrhage.pulse_hemorrhage` | 600 | state_file=Underweight@0s.json |
| `Ventilated` | `viva_pulse.composites.hemorrhage.pulse_hemorrhage` | 600 | state_file=Ventilated@0s.json |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_pulse.composites.hemorrhage.pulse_hemorrhage`**

Untreated hemorrhage in the real Pulse whole-body adult engine; emits vital signs every second.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage = {
    'state_file'       : 'StandardMale@0s.json',   # Stabilized Pulse patient state in <install>/bin/states
    'patient_file'     : 'StandardMale.json',   # Any patient instead of state_file: a name in <install>/bin/patients or a path; stabilized once (about 13 s), then cached
    'circuit_modifiers': {},   # Circuit path or node name -> multiplier on the value Pulse initializes it with; the patient is stabilized with them (about 25 s the first time, then cached)
    'circuit_additions': {},   # New path name -> {source, target, resistance_mmHg_s_per_mL}: a cardiovascular path added with its compartment link, as a perturbation of the loaded patient
    'compartment'      : 'RightLeg',   # Hemorrhage site (Pulse eHemorrhage_Compartment)
    'severity'         : 0.8,   # Hemorrhage severity in [0, 1]
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: patient-cohort ===
STUDY = 'patient-cohort'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Reproduce the recorded run(s). STEPS = composite steps.
STEPS_Bradycardic = 600
INTERVAL_Bradycardic = 1.0
STEPS_Carol = 600
INTERVAL_Carol = 1.0
STEPS_Cynthia = 600
INTERVAL_Cynthia = 1.0
STEPS_DefaultFemale = 600
INTERVAL_DefaultFemale = 1.0
STEPS_DefaultMale = 600
INTERVAL_DefaultMale = 1.0
STEPS_ExtremeFemale = 600
INTERVAL_ExtremeFemale = 1.0
STEPS_ExtremeMale = 600
INTERVAL_ExtremeMale = 1.0
STEPS_Gus = 600
INTERVAL_Gus = 1.0
STEPS_Hassan = 600
INTERVAL_Hassan = 1.0
STEPS_Jane = 600
INTERVAL_Jane = 1.0
STEPS_Jeff = 600
INTERVAL_Jeff = 1.0
STEPS_Joel = 600
INTERVAL_Joel = 1.0
STEPS_Nathan = 600
INTERVAL_Nathan = 1.0
STEPS_Overweight = 600
INTERVAL_Overweight = 1.0
STEPS_Rick = 600
INTERVAL_Rick = 1.0
STEPS_Soldier = 600
INTERVAL_Soldier = 1.0
STEPS_StandardFemale = 600
INTERVAL_StandardFemale = 1.0
STEPS_StandardMale = 600
INTERVAL_StandardMale = 1.0
STEPS_Tachycardic = 600
INTERVAL_Tachycardic = 1.0
STEPS_Underweight = 600
INTERVAL_Underweight = 1.0
STEPS_Ventilated = 600
INTERVAL_Ventilated = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage if 'ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage = materialize_composite('viva_pulse.composites.hemorrhage.pulse_hemorrhage', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage['state']}, core=core)
            _comp.run(STEPS_Bradycardic)
        print('ran', 'Bradycardic', f'(viva_pulse.composites.hemorrhage.pulse_hemorrhage) for', STEPS_Bradycardic, 'steps')
    except Exception as _e:
        print(f"Bradycardic not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage if 'ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage = materialize_composite('viva_pulse.composites.hemorrhage.pulse_hemorrhage', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage['state']}, core=core)
            _comp.run(STEPS_Carol)
        print('ran', 'Carol', f'(viva_pulse.composites.hemorrhage.pulse_hemorrhage) for', STEPS_Carol, 'steps')
    except Exception as _e:
        print(f"Carol not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage if 'ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage = materialize_composite('viva_pulse.composites.hemorrhage.pulse_hemorrhage', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage['state']}, core=core)
            _comp.run(STEPS_Cynthia)
        print('ran', 'Cynthia', f'(viva_pulse.composites.hemorrhage.pulse_hemorrhage) for', STEPS_Cynthia, 'steps')
    except Exception as _e:
        print(f"Cynthia not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage if 'ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage = materialize_composite('viva_pulse.composites.hemorrhage.pulse_hemorrhage', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage['state']}, core=core)
            _comp.run(STEPS_DefaultFemale)
        print('ran', 'DefaultFemale', f'(viva_pulse.composites.hemorrhage.pulse_hemorrhage) for', STEPS_DefaultFemale, 'steps')
    except Exception as _e:
        print(f"DefaultFemale not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage if 'ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage = materialize_composite('viva_pulse.composites.hemorrhage.pulse_hemorrhage', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage['state']}, core=core)
            _comp.run(STEPS_DefaultMale)
        print('ran', 'DefaultMale', f'(viva_pulse.composites.hemorrhage.pulse_hemorrhage) for', STEPS_DefaultMale, 'steps')
    except Exception as _e:
        print(f"DefaultMale not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage if 'ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage = materialize_composite('viva_pulse.composites.hemorrhage.pulse_hemorrhage', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage['state']}, core=core)
            _comp.run(STEPS_ExtremeFemale)
        print('ran', 'ExtremeFemale', f'(viva_pulse.composites.hemorrhage.pulse_hemorrhage) for', STEPS_ExtremeFemale, 'steps')
    except Exception as _e:
        print(f"ExtremeFemale not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage if 'ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage = materialize_composite('viva_pulse.composites.hemorrhage.pulse_hemorrhage', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage['state']}, core=core)
            _comp.run(STEPS_ExtremeMale)
        print('ran', 'ExtremeMale', f'(viva_pulse.composites.hemorrhage.pulse_hemorrhage) for', STEPS_ExtremeMale, 'steps')
    except Exception as _e:
        print(f"ExtremeMale not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage if 'ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage = materialize_composite('viva_pulse.composites.hemorrhage.pulse_hemorrhage', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage['state']}, core=core)
            _comp.run(STEPS_Gus)
        print('ran', 'Gus', f'(viva_pulse.composites.hemorrhage.pulse_hemorrhage) for', STEPS_Gus, 'steps')
    except Exception as _e:
        print(f"Gus not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage if 'ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage = materialize_composite('viva_pulse.composites.hemorrhage.pulse_hemorrhage', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage['state']}, core=core)
            _comp.run(STEPS_Hassan)
        print('ran', 'Hassan', f'(viva_pulse.composites.hemorrhage.pulse_hemorrhage) for', STEPS_Hassan, 'steps')
    except Exception as _e:
        print(f"Hassan not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage if 'ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage = materialize_composite('viva_pulse.composites.hemorrhage.pulse_hemorrhage', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage['state']}, core=core)
            _comp.run(STEPS_Jane)
        print('ran', 'Jane', f'(viva_pulse.composites.hemorrhage.pulse_hemorrhage) for', STEPS_Jane, 'steps')
    except Exception as _e:
        print(f"Jane not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage if 'ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage = materialize_composite('viva_pulse.composites.hemorrhage.pulse_hemorrhage', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage['state']}, core=core)
            _comp.run(STEPS_Jeff)
        print('ran', 'Jeff', f'(viva_pulse.composites.hemorrhage.pulse_hemorrhage) for', STEPS_Jeff, 'steps')
    except Exception as _e:
        print(f"Jeff not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage if 'ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage = materialize_composite('viva_pulse.composites.hemorrhage.pulse_hemorrhage', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage['state']}, core=core)
            _comp.run(STEPS_Joel)
        print('ran', 'Joel', f'(viva_pulse.composites.hemorrhage.pulse_hemorrhage) for', STEPS_Joel, 'steps')
    except Exception as _e:
        print(f"Joel not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage if 'ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage = materialize_composite('viva_pulse.composites.hemorrhage.pulse_hemorrhage', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage['state']}, core=core)
            _comp.run(STEPS_Nathan)
        print('ran', 'Nathan', f'(viva_pulse.composites.hemorrhage.pulse_hemorrhage) for', STEPS_Nathan, 'steps')
    except Exception as _e:
        print(f"Nathan not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage if 'ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage = materialize_composite('viva_pulse.composites.hemorrhage.pulse_hemorrhage', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage['state']}, core=core)
            _comp.run(STEPS_Overweight)
        print('ran', 'Overweight', f'(viva_pulse.composites.hemorrhage.pulse_hemorrhage) for', STEPS_Overweight, 'steps')
    except Exception as _e:
        print(f"Overweight not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage if 'ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage = materialize_composite('viva_pulse.composites.hemorrhage.pulse_hemorrhage', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage['state']}, core=core)
            _comp.run(STEPS_Rick)
        print('ran', 'Rick', f'(viva_pulse.composites.hemorrhage.pulse_hemorrhage) for', STEPS_Rick, 'steps')
    except Exception as _e:
        print(f"Rick not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage if 'ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage = materialize_composite('viva_pulse.composites.hemorrhage.pulse_hemorrhage', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage['state']}, core=core)
            _comp.run(STEPS_Soldier)
        print('ran', 'Soldier', f'(viva_pulse.composites.hemorrhage.pulse_hemorrhage) for', STEPS_Soldier, 'steps')
    except Exception as _e:
        print(f"Soldier not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage if 'ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage = materialize_composite('viva_pulse.composites.hemorrhage.pulse_hemorrhage', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage['state']}, core=core)
            _comp.run(STEPS_StandardFemale)
        print('ran', 'StandardFemale', f'(viva_pulse.composites.hemorrhage.pulse_hemorrhage) for', STEPS_StandardFemale, 'steps')
    except Exception as _e:
        print(f"StandardFemale not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage if 'ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage = materialize_composite('viva_pulse.composites.hemorrhage.pulse_hemorrhage', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage['state']}, core=core)
            _comp.run(STEPS_StandardMale)
        print('ran', 'StandardMale', f'(viva_pulse.composites.hemorrhage.pulse_hemorrhage) for', STEPS_StandardMale, 'steps')
    except Exception as _e:
        print(f"StandardMale not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage if 'ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage = materialize_composite('viva_pulse.composites.hemorrhage.pulse_hemorrhage', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage['state']}, core=core)
            _comp.run(STEPS_Tachycardic)
        print('ran', 'Tachycardic', f'(viva_pulse.composites.hemorrhage.pulse_hemorrhage) for', STEPS_Tachycardic, 'steps')
    except Exception as _e:
        print(f"Tachycardic not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage if 'ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage = materialize_composite('viva_pulse.composites.hemorrhage.pulse_hemorrhage', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage['state']}, core=core)
            _comp.run(STEPS_Underweight)
        print('ran', 'Underweight', f'(viva_pulse.composites.hemorrhage.pulse_hemorrhage) for', STEPS_Underweight, 'steps')
    except Exception as _e:
        print(f"Underweight not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage if 'ov_viva_pulse_composites_hemorrhage_pulse_hemorrhage' in dir() else None  # edits from the panel cell, if any
            spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage = materialize_composite('viva_pulse.composites.hemorrhage.pulse_hemorrhage', _ov)
            _comp = Composite({'state': spec_viva_pulse_composites_hemorrhage_pulse_hemorrhage['state']}, core=core)
            _comp.run(STEPS_Ventilated)
        print('ran', 'Ventilated', f'(viva_pulse.composites.hemorrhage.pulse_hemorrhage) for', STEPS_Ventilated, 'steps')
    except Exception as _e:
        print(f"Ventilated not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**cohort-blood-loss**


In [ ]:
# cohort-blood-loss
show_viz(_render_one('local:CohortBloodLoss', {'inputs_map': {'blood_volume': 'vitals.blood_volume_mL', 'hemorrhaged_volume': 'vitals.total_hemorrhaged_volume_mL', 'mean_arterial_pressure': 'vitals.mean_arterial_pressure_mmHg'}}, RUNS_DB, STUDY_YAML))